# Phase 14.1 — RAG Document Ingestion

This notebook creates and ingests synthetic business documents
for the GenAI Data Analyst Copilot.

Documents are intentionally synthetic and contain no real
personal or confidential information.

Documents:

- Discount Policy
- Return Policy
- Regional Sales Guidelines
- Product Policy

The documents will be stored in Unity Catalog as a Delta table
for downstream RAG processing.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

from datetime import datetime
import uuid

In [0]:
RAG_SCHEMA = "genai_copilot.gold"

DOCUMENT_TABLE = "genai_copilot.gold.business_documents"

print("RAG schema:", RAG_SCHEMA)
print("Document table:", DOCUMENT_TABLE)

In [0]:
documents = [
    {
        "document_id": "DOC001",
        "file_name": "discount_policy.txt",
        "document_type": "policy",
        "title": "Discount Policy",
        "content": """
Discount Policy

Standard discounts may be offered based on customer segment,
order quantity, sales channel, and approved promotional campaigns.

Retail customers may receive discounts up to 10 percent during
approved promotional periods.

Enterprise customers may receive discounts up to 15 percent
when the order meets the minimum volume requirement.

Discounts above the standard thresholds require approval from
the regional sales manager.

Sales representatives must record the approved discount in the
sales system before the order is completed.

Discounts cannot be combined unless explicitly approved by the
commercial operations team.
"""
    },
    {
        "document_id": "DOC002",
        "file_name": "return_policy.txt",
        "document_type": "policy",
        "title": "Return Policy",
        "content": """
Return Policy

Customers may request a return within 30 calendar days of
delivery for eligible products.

Products must be returned in acceptable condition unless the
return is caused by a manufacturing defect.

Approved returns are recorded against the original order.

Refunds are normally processed after the returned product has
been inspected.

Products marked as final sale are not eligible for standard
returns.

Exceptions may be approved by customer support management.
"""
    },
    {
        "document_id": "DOC003",
        "file_name": "regional_sales_guidelines.txt",
        "document_type": "guideline",
        "title": "Regional Sales Guidelines",
        "content": """
Regional Sales Guidelines

Sales teams should monitor revenue, order volume, profit margin,
and customer retention by region.

Regional managers are responsible for reviewing significant
revenue declines.

Promotional campaigns should consider local market conditions,
customer demand, and approved pricing policies.

Regional teams must follow the global discount policy when
offering customer discounts.

Exceptions to standard pricing require documented approval.
"""
    },
    {
        "document_id": "DOC004",
        "file_name": "product_policy.txt",
        "document_type": "policy",
        "title": "Product Policy",
        "content": """
Product Policy

Products are categorized according to the organization's
product hierarchy.

Product pricing must be maintained by authorized product
management teams.

Discontinued products should not be included in new customer
orders.

Product substitutions require customer notification when the
replacement materially differs from the original product.

Product managers should periodically review product performance,
profitability, and customer demand.
"""
    }
]

print("Documents created:", len(documents))

In [0]:
schema = StructType([
    StructField("document_id", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("document_type", StringType(), False),
    StructField("title", StringType(), False),
    StructField("content", StringType(), False),
    StructField("ingestion_timestamp", TimestampType(), False)
])

rows = []

for document in documents:

    rows.append(
        (
            document["document_id"],
            document["file_name"],
            document["document_type"],
            document["title"],
            document["content"].strip(),
            datetime.now()
        )
    )

documents_df = spark.createDataFrame(
    rows,
    schema=schema
)

display(documents_df)

In [0]:
documents_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(DOCUMENT_TABLE)

print("Document table created:")
print(DOCUMENT_TABLE)

In [0]:
display(
    spark.table(DOCUMENT_TABLE)
    .select(
        "document_id",
        "file_name",
        "document_type",
        "title"
    )
    .orderBy("document_id")
)

In [0]:
%sql
SELECT
    document_id,
    file_name,
    document_type,
    title,
    length(content) AS content_length
FROM genai_copilot.gold.business_documents
ORDER BY document_id;

14.11 Why store documents in Delta?

This is an important Data Engineering aspect of our RAG architecture.

Instead of:

PDF → directly into LLM

we're building:

Documents
    ↓
Bronze-like document storage
    ↓
Structured Delta table
    ↓
Chunking
    ↓
Retrieval
    ↓
LLM

That gives us:

reproducibility
metadata
auditability
versionable processing
integration with Unity Catalog
easier evaluation